In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_mistralai import ChatMistralAI
from langchain_core.tools import tool
from dotenv import load_dotenv
import os 
import requests
from pydantic import BaseModel
from typing import Annotated
from langgraph.graph import add_messages
from langgraph.prebuilt import ToolNode,tools_condition
from IPython.display import Markdown,display
from langchain_tavily import TavilySearch


In [2]:
llm=ChatMistralAI(
    model="open-mistral-7b"
)

In [3]:
class State(BaseModel):
    messages:Annotated[list,add_messages]

In [4]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city using OpenWeather API."""
    print("weather tool")
    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": city,
        "appid": os.getenv("OPENWEATHER_API_KEY"),
        "units": "metric"
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        return f"Could not get weather for {city}."

    data = response.json()

    temperature = data["main"]["temp"]
    feels_like = data["main"]["feels_like"]
    humidity = data["main"]["humidity"]
    description = data["weather"][0]["description"]

    return (
        f"Weather in {city}: "
        f"{description}, "
        f"temperature {temperature}°C, "
        f"feels like {feels_like}°C, "
        f"humidity {humidity}%."
    )

In [5]:
@tool
def send_notification(message: str) -> str:
    """Send a notification to the user through Pushover."""
    print("notification send")
    url = "https://api.pushover.net/1/messages.json"

    data = {
        "token": os.getenv("PUSHOVER_TOKEN"),
        "user": os.getenv("PUSHOVER_USER"),
        "message": message,
    }

    response = requests.post(url, data=data)

    if response.status_code != 200:
        return f"Failed to send notification: {response.text}"

    result = response.json()

    if result.get("status") == 1:
        return "Notification sent successfully."

    return f"Failed to send notification: {result}"

In [6]:
search_tool=TavilySearch(max_results=3)

In [7]:


tools=[get_weather,send_notification,search_tool]
llm_with_tools=llm.bind_tools(tools)

In [8]:
def chatbot(state:State):
    result=llm_with_tools.invoke(state.messages)
    return {"messages":[result]}

In [9]:
tool_node=ToolNode(tools)

In [10]:
graph_builder=StateGraph(State)

In [11]:
graph_builder.add_node("chatbot",chatbot)
graph_builder.add_node("tools",tool_node)

In [12]:
graph_builder.add_edge(START,"chatbot")
graph_builder.add_conditional_edges("chatbot",tools_condition)
graph_builder.add_edge("tools","chatbot")
graph_builder.add_edge("chatbot",END)

In [13]:
graph=graph_builder.compile()

In [25]:
query=input("")
initial_state={
    "messages": [
        ("user", query)
    ]
}

In [27]:
result = graph.invoke(initial_state)


notification send


In [16]:
print(result["messages"][-1].content)

The current weather in **Islamabad** is **clear sky** with a temperature of **30.47°C**, but it feels like **33.75°C** due to humidity. The humidity level is **60%**.

Would you like any additional details or alerts? 😊


In [28]:
for i in result["messages"]:
    i.pretty_print()
print()

================================ Human Message =================================

send me notification about the future of agentic ai
================================== Ai Message ==================================
Tool Calls:
  tavily_search (UNmGOz8lc)
 Call ID: UNmGOz8lc
  Args:
    query: future of agentic AI 2024 to 2030
    search_depth: advanced
    time_range: year
    topic: general
================================= Tool Message =================================
Name: tavily_search

{"query": "future of agentic AI 2024 to 2030", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://omdia.tech.informa.com/pr/2025/sep/new-omdia-analysis-shows-agentic-ai-outpacing-growth-rates-of-traditional-generative-ai", "title": "New Omdia analysis shows Agentic AI outpacing growth rates of traditional generative AI", "content": "Enterprise agentic AI software revenue forecast, 2024-2030\n\nEnterprise agentic AI software revenue forecast, 2024-2030\n\nEnterpr